Goals of Data Cleaning (From README.md)

What needs to be checked 

    - no missing values (impute)
    - correct data types per column 
    - ensuring correct units per column (for sales - ensure the calculation is correct- no negative values, no outlandish values)
    - outliers 
    - Merge columns that waste space having seperate features (feature selection/engineering)

In [1]:
import pandas as pd 
import numpy as np 


In [2]:
!pip install openpyxl

In [7]:
df = pd.read_excel('C:/Users/jagmeet/vsc/data/pharm_data.xlsx', sheet_name='Data')

In [8]:
df.shape
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 254082 entries, 0 to 254081
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Distributor        254082 non-null  str    
 1   Customer Name      254082 non-null  str    
 2   City               254082 non-null  str    
 3   Country            254082 non-null  str    
 4   Latitude           254082 non-null  float64
 5   Longitude          254082 non-null  float64
 6   Channel            254082 non-null  str    
 7   Sub-channel        254082 non-null  str    
 8   Product Name       254082 non-null  str    
 9   Product Class      254082 non-null  str    
 10  Quantity           254082 non-null  float64
 11  Price              254082 non-null  int64  
 12  Sales              254082 non-null  float64
 13  Month              254082 non-null  str    
 14  Year               254082 non-null  int64  
 15  Name of Sales Rep  254082 non-null  str    
 16  Manager      

In [ ]:
# no missing values
df.isna().sum()

Distributor          0
Customer Name        0
City                 0
Country              0
Latitude             0
Longitude            0
Channel              0
Sub-channel          0
Product Name         0
Product Class        0
Quantity             0
Price                0
Sales                0
Month                0
Year                 0
Name of Sales Rep    0
Manager              0
Sales Team           0
dtype: int64

In [ ]:
# correct data types per column 
df['Quantity'].max()
df['Quantity'].min() # -7200 - must be a mistake in score

#df.info()
print(df['Price'].max())
print(df['Price'].min()) 

print(df['Sales'].max())
print(df['Sales'].min()) 
# quantity carries over to sales as sales = quantity x price (incorrect negative value)

#get all values which are negative
df[df['Quantity'] < 0].sum # 2633  values have negative quantity values thus negative sales values.


794
22
74205600.0
-4161600.0


<bound method DataFrame.sum of             Distributor                                      Customer Name  \
3761           Rohan                               Zieme, Doyle and Kunze    
3762           Rohan                  Nader-Gaylord Pharmaceutical Limited   
18526          Rohan     Buckridge, Dach and Carroll Pharmaceutical Lim...   
22613   Prohaska-Kuhic                   Wiegand, Jast and Yost Pharma Plc   
22751   Prohaska-Kuhic               Pfeffer-Hodkiewicz Pharmaceutical Ltd   
...                 ...                                                ...   
253760          Koss                                 Murray PLC Pharma Plc   
253761          Koss                                          Kreiger Inc    
253762          Koss                 Gerhold, Mills and Effertz Pharma Plc   
253763          Koss                               Pfeffer-Kirlin Pharmacy   
253998          Koss                                 Corkery-Kovacek Pharm   

              City  Country  Lat

In [ ]:
return_df = df[df['Quantity'] < 0] # 2633 - exclude and do a seperate negative analysis on them 
sales_df = df[df['Quantity'] >= 0] # new dataset includes samples which have positive quantity values


Checking for more Outliers:

In [53]:
sales_df

,Distributor,Customer Name,City,Country,Latitude,Longitude,Channel,Sub-channel,Product Name,Product Class,Quantity,Price,Sales,Month,Year,Name of Sales Rep,Manager,Sales Team
0,Gottlieb-Cruickshank,"Zieme, Doyle and Kunze",Lublin,Poland,51.2333,22.5667,Hospital,Private,Topipizole,Mood Stabilizers,4.0,368,1472.0,January,2018,Mary Gerrard,Britanny Bold,Delta
1,Gottlieb-Cruickshank,Feest PLC,Świecie,Poland,53.4167,18.4333,Pharmacy,Retail,Choriotrisin,Antibiotics,7.0,591,4137.0,January,2018,Jessica Smith,Britanny Bold,Delta
2,Gottlieb-Cruickshank,Medhurst-Beer Pharmaceutical Limited,Rybnik,Poland,50.0833,18.5000,Pharmacy,Institution,Acantaine,Antibiotics,30.0,66,1980.0,January,2018,Steve Pepple,Tracy Banks,Bravo
3,Gottlieb-Cruickshank,Barton Ltd Pharma Plc,Czeladź,Poland,50.3333,19.0833,Hospital,Private,Lioletine Refliruvax,Analgesics,6.0,435,2610.0,January,2018,Mary Gerrard,Britanny Bold,Delta
4,Gottlieb-Cruickshank,Keeling LLC Pharmacy,Olsztyn,Poland,53.7800,20.4942,Pharmacy,Retail,Oxymotroban Fexoformin,Analgesics,20.0,458,9160.0,January,2018,Anne Wu,Britanny Bold,Delta
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254077,Bashirian-Kassulke,"Koch, Borer and Hagenes Pharmaceutical Ltd",Lauf,Germany,49.5103,11.2772,Hospital,Private,Pentastrin,Antibiotics,919.0,497,456743.0,December,2020,Thompson Crawford,James Goodwill,Alfa
254078,Bashirian-Kassulke,Hane Ltd Pharmaceutical Ltd,Aichach,Germany,48.4500,11.1333,Hospital,Private,Abranatal Lysoprosate,Antiseptics,432.0,681,294192.0,December,2020,Anne Wu,Britanny Bold,Delta
254079,Bashirian-Kassulke,Harris-Conroy Pharmacy,Wilhelmshaven,Germany,53.5167,8.1333,Pharmacy,Retail,Adideine,Mood Stabilizers,320.0,678,216960.0,December,2020,Abigail Thompson,Tracy Banks,Bravo
254080,Bashirian-Kassulke,Balistreri Group Pharm,Böblingen,Germany,48.6833,9.0000,Hospital,Government,Feruprazole,Mood Stabilizers,565.0,115,64975.0,December,2020,Stella Given,Alisha Cordwell,Charlie


In [58]:
def count_outliers(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3-q1

    lower_bound = q1 - (1.5*iqr)
    upper_bound = q3 + (1.5*iqr)

    #filtering the outliers:

    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    print(f" the number of outliers in column: {len(outliers)}")
    return outliers

sales_outliers = count_outliers(sales_df, 'Sales')
qty_outliers = count_outliers(sales_df, 'Quantity')


 the number of outliers in column: 34876
 the number of outliers in column: 38074


In [ ]:
# check if the slaes calculation is correct:
sales_df['calc_sales'] = sales_df['Quantity'] * sales_df['Price']
 
sales_df[sales_df['calc_sales'] != sales_df['Sales']]
# tehre are no rows which do not match up which means the calculation is correct.



,Distributor,Customer Name,City,Country,Latitude,Longitude,Channel,Sub-channel,Product Name,Product Class,Quantity,Price,Sales,Month,Year,Name of Sales Rep,Manager,Sales Team,calc_sales


In [63]:
# check which managers have the negative quantity prices:
df[df['Quantity']<0]['Manager'].value_counts()

Manager
Britanny Bold      792
James Goodwill     627
Tracy Banks        623
Alisha Cordwell    591
Name: count, dtype: int64